# Zyro AI/ML Internship • Week 3
## Task 02: Improve Document Understanding
### Document Classification Benchmark & Evaluation

This interactive notebook demonstrates the end-to-end Machine Learning pipeline for document understanding:
1. **Dataset Loading**: Ingestion of a curated, balanced 36-document corpus (Invoice, Resume, Other).
2. **Text Cleaning & Preprocessing**: Unicode normalization, space trimming, and token preparation.
3. **Feature Engineering**: TF-IDF Vectorization with unigrams and bigrams.
4. **Model Comparison**: Logistic Regression, Calibrated Linear SVM, Multinomial Naive Bayes, and Rule-Based Baseline.
5. **Evaluation**: 4-Fold Stratified Cross-Validation, Classification Report (Accuracy, Precision, Recall, F1), and Confusion Matrices.
6. **Confidence Calibration**: Platt Scaling to generate reliable class confidence percentages.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Custom text cleaner
from text_cleaner import clean_extracted_text

print("All dependencies imported successfully!")

### 1. Load Dataset Corpus

In [ ]:
corpus_path = 'dataset/documents_corpus.json'
with open(corpus_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

df_docs = pd.DataFrame(data)
df_docs['cleaned_text'] = df_docs['text'].apply(clean_extracted_text)

print(f"Total documents in corpus: {len(df_docs)}")
print("\nClass Distribution:")
print(df_docs['category'].value_counts())

### 2. Feature Extraction (TF-IDF Vectorization)

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=1, lowercase=True)
X_tfidf = vectorizer.fit_transform(df_docs['cleaned_text'])
print(f"TF-IDF Feature Space Dimensions: {X_tfidf.shape}")

### 3. Model Benchmark & Evaluation

In [ ]:
from model_trainer import build_candidate_models, CLASSES

X = df_docs['cleaned_text'].tolist()
y = df_docs['category'].tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
models = build_candidate_models()

results = []
confusion_matrices = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    confusion_matrices[name] = confusion_matrix(y_test, y_pred, labels=CLASSES)
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })

df_results = pd.DataFrame(results)
display(df_results)

### 4. Visualizing Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5), dpi=150)
for ax, (name, cm) in zip(axes, confusion_matrices.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax,
                annot_kws={'size': 12, 'weight': 'bold'})
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted Class')
    ax.set_ylabel('True Class')

plt.suptitle('Confusion Matrices Comparison (Week 3)', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

### 5. Calibrated Confidence Scoring Test

In [ ]:
from model_trainer import classify_document

test_doc = "TAX INVOICE Apex Digital Solutions LLC Invoice Number: INV-2026-0814 Total Amount: $8,389.38"
res = classify_document(test_doc)
print("Prediction:", res['document_type'])
print("Calibrated Confidence:", res['confidence_percentage'])
print("Class Probabilities:", res['probabilities'])